# Drishti M1: accident detector (YOLO11n)

Fine-tunes YOLO11n on the Roboflow Universe dataset `ann-qwhjm/accident-detection-cwbvs` v2 (classes accident / moderate / severe; the dataset's own `data.yaml` states CC BY 4.0). Goal: our own nano-size accident weights, so the project does not depend on an unlicensed `.pt`.

**Before running:** Settings -> Accelerator -> GPU T4, Internet on, and add a Kaggle secret `ROBOFLOW_API_KEY` (Add-ons -> Secrets).

In [ ]:
!pip -q install ultralytics==8.4.168 roboflow

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

rf = Roboflow(api_key=UserSecretsClient().get_secret('ROBOFLOW_API_KEY'))
ds = rf.workspace('ann-qwhjm').project('accident-detection-cwbvs').version(2).download('yolov11')
print(ds.location)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(data=f'{ds.location}/data.yaml', epochs=80, imgsz=640, batch=32, patience=15,
            project='runs', name='drishti', seed=0)

## Held-out metrics

Runs on the dataset's **test** split, which training never saw. Copy `metrics.json` into `docs/results/` in the repo.

In [ ]:
import json

best = YOLO('runs/drishti/weights/best.pt')
m = best.val(data=f'{ds.location}/data.yaml', split='test', imgsz=640)
metrics = {'dataset': 'ann-qwhjm/accident-detection-cwbvs v2', 'split': 'test',
           'map50': float(m.box.map50), 'map50_95': float(m.box.map),
           'precision': float(m.box.mp), 'recall': float(m.box.mr),
           'classes': best.names, 'images': int(m.seen) if hasattr(m, 'seen') else None}
json.dump(metrics, open('accident_metrics.json', 'w'), indent=1)
metrics

In [ ]:
import shutil
shutil.copy('runs/drishti/weights/best.pt', 'accident.pt')
print('Download accident.pt and accident_metrics.json from the Output panel')

## Using the weights

1. Put `accident.pt` in the repo's `models/` folder (replacing the current one).
2. Run `backend\.venv\Scripts\python training\bench\run_bench.py accident_weights ours_yolo11n models\accident.pt`
   then `training\bench\summarise.py ours_yolo11n` and compare with BENCH.md.
3. Keep the new weights only if they beat the current model on our clips. Then rerun `training\eval_e2e.py`, `training\prepare_demo.py` and `training\bench\make_summary.py`.